# QuALITY full-dev confirmation on Colab

This notebook runs the Qwen2.5-1.5B confirmation on all 2,086 labelled QuALITY dev questions. It uses a segmented uncached reference once per workload, then compares document, fixed-block-256, radix, CPU FP16, CPU INT8/Triton, and GDSF where each comparison is informative.

The 12 runs are split into four checkpoints. On a T4, expect roughly 12--18 GPU-hours in total; individual runtimes vary. Every command is resumable at completed-run granularity. Smoke timings are functional checks only and must not be reported as evidence.

In [1]:
import os

# Set allocator behavior before importing torch.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
!nvidia-smi

Sat Sep  5 07:13:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Clone or update the repository

In [2]:
from pathlib import Path

repo = Path("/content/rag-kvcache")
!test -d /content/rag-kvcache || git clone https://github.com/i0nut02/rag-kvcache.git /content/rag-kvcache
os.chdir(repo)
!git pull --ff-only
print(Path.cwd())

Cloning into '/content/rag-kvcache'...
remote: Enumerating objects: 456, done.
remote: Counting objects: 100% (456/456), done.
remote: Compressing objects: 100% (298/298), done.
remote: Total 456 (delta 270), reused 335 (delta 154), pack-reused 0 (from 0)
Receiving objects: 100% (456/456), 616.78 KiB | 2.49 MiB/s, done.
Resolving deltas: 100% (270/270), done.
Already up to date.
/content/rag-kvcache


In [3]:
import torch
import triton

assert torch.cuda.is_available(), "Select a GPU runtime before continuing"
print("Torch:", torch.__version__)
print("Triton:", triton.__version__)
print("GPU:", torch.cuda.get_device_name(0))

Torch: 2.11.0+cu128
Triton: 3.6.0
GPU: Tesla T4


In [4]:
# Optional: add HF_TOKEN as a Colab secret to avoid anonymous Hub limits.
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
print("HF token configured:", bool(hf_token))

HF token configured: False


## 2. Download and validate QuALITY dev

Only the labelled dev file is needed for this suite.

In [5]:
data_dir = Path("data/quality-v1.0.1")
data_dir.mkdir(parents=True, exist_ok=True)
!wget -q -nc -P {data_dir} https://raw.githubusercontent.com/nyu-mll/quality/main/data/v1.0.1/QuALITY.v1.0.1.htmlstripped.dev
!python experiments/run_quality.py validate-data data/quality-v1.0.1/QuALITY.v1.0.1.htmlstripped.dev --split dev --verify-counts

{
  "split": "dev",
  "articles": 115,
  "questions": 2086,
  "grouped_max_hit_rate": 0.9448705656759347,
  "sha256": "99852d874994078e4b4112b71ceca4dd35aa3a24ff6d3a35c051be25295b4fef"
}


In [6]:
# Focused checks for reference alignment and tensor storage. Matrix expansion below validates the new config.
!python -m unittest tests.test_reference tests.test_tensors -q
!python experiments/run_quality.py matrix configs/full_dev_confirmation.json --profile full --show-commands

----------------------------------------------------------------------
Ran 17 tests in 3.403s

OK (skipped=1)
{
  "dataset": "data/quality-v1.0.1/QuALITY.v1.0.1.htmlstripped.dev",
  "split": "dev",
  "profile": "full",
  "no_inference": false,
  "matrix_type": "selected",
  "combinations": 12,
  "output_dir": "results/full_dev_confirmation/full",
  "execute": false
}
python experiments/run_quality.py run data/quality-v1.0.1/QuALITY.v1.0.1.htmlstripped.dev --split dev --verify-counts --model Qwen/Qwen2.5-1.5B-Instruct --tokenizer Qwen/Qwen2.5-1.5B-Instruct --device cuda --dtype float16 --cache-strategy document --policy none --storage accelerator-fp16 --workload random --seed 42 --block-tokens 16 --baseline-mode segmented --agreement-atol 0.0625 --progress-every 50 --output results/full_dev_confirmation/full/dev_full_01_segmented_random_fp16.jsonl
python experiments/run_quality.py run data/quality-v1.0.1/QuALITY.v1.0.1.htmlstripped.dev --split dev --verify-counts --model Qwen/Qwen2.5-1.

## 3. Optional checkpoint restore

If a previous Colab runtime expired, upload the most recent `full-dev-checkpoint-*.zip` from your computer and call `restore_checkpoint()`. The ZIP is unpacked into the exact result directory used by `--resume`. Do not rename or edit files inside the archive.

In [7]:
import shutil
from google.colab import files

full_results = Path("results/full_dev_confirmation/full")
full_results.mkdir(parents=True, exist_ok=True)

def restore_checkpoint():
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one checkpoint ZIP")
    archive = Path(next(iter(uploaded)))
    shutil.unpack_archive(archive, full_results)
    print("Restored:", archive, "to", full_results)

def download_checkpoint(label, source=full_results):
    source = Path(source)
    archive = shutil.make_archive(f"/content/{label}", "zip", root_dir=source)
    print("Created:", archive, f"({Path(archive).stat().st_size / 2**20:.1f} MiB)")
    files.download(archive)

# Uncomment only when restoring a downloaded checkpoint in a new runtime.
restore_checkpoint()
restore_checkpoint()
restore_checkpoint()

Saving full-dev-checkpoint-06-random-complete.zip to full-dev-checkpoint-06-random-complete.zip
Restored: full-dev-checkpoint-06-random-complete.zip to results/full_dev_confirmation/full


Saving full-dev-checkpoint-09.zip to full-dev-checkpoint-09.zip
Restored: full-dev-checkpoint-09.zip to results/full_dev_confirmation/full


ValueError: Upload exactly one checkpoint ZIP

## 4. Ten-request functional smoke

This exercises all 12 paths, including Triton, before spending hours. It writes to a separate directory and does not affect the full profile.

In [8]:
!python experiments/run_quality.py matrix configs/full_dev_confirmation.json --profile smoke --execute --resume

{
  "dataset": "data/quality-v1.0.1/QuALITY.v1.0.1.htmlstripped.dev",
  "split": "dev",
  "profile": "smoke",
  "no_inference": false,
  "matrix_type": "selected",
  "combinations": 12,
  "output_dir": "results/full_dev_confirmation/smoke",
  "execute": true
}
[1/12] run dev_smoke_01_segmented_random_fp16.jsonl
config.json: 100% 660/660 [00:00<00:00, 3.50MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 23.9MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 65.8MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 88.6MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 138MB/s]

model.safetensors: downloading bytes:   0% 0.00/3.09G [00:00<?, ?B/s]
model.safetensors: downloading bytes:   2% 53.6M/3.09G [00:01<00:36, 83.2MB/s, 3.27MB/s  ]
model.safetensors: downloading bytes:   3% 80.9M/3.09G [00:01<00:23, 126MB/s, 5.20MB/s  ] 
model.safetensors: downloading bytes:   5% 158M/3.09G [00:01<00:13, 209MB/s, 11.9MB/s  ]
model.safetensors: reconstructing file:   4% 137M/3.09G [00:01<00:22,

In [9]:
import json
import pandas as pd

def show_summaries(profile="full"):
    rows = []
    for path in sorted(Path(f"results/full_dev_confirmation/{profile}").glob("*.summary.json")):
        row = json.loads(path.read_text())
        row["run"] = path.name.removesuffix(".summary.json")
        rows.append(row)
    columns = [
        "run", "requests", "workload", "cache_strategy", "policy",
        "storage", "ttft_mean_s", "ttft_p95_s",
        "article_token_hit_rate", "accuracy",
        "reference_label_agreement", "reference_label_mismatches",
    ]
    return pd.DataFrame(rows).reindex(columns=columns)

show_summaries("smoke")

,run,requests,workload,cache_strategy,policy,storage,ttft_mean_s,ttft_p95_s,article_token_hit_rate,accuracy,reference_label_agreement,reference_label_mismatches
0,dev_smoke_01_segmented_random_fp16,10,random,none,none,accelerator-fp16,2.039375,2.566651,0.000000,0.4,NaN,0
1,dev_smoke_02_document_lru_random_fp16_4gib,10,random,document,lru,accelerator-fp16,2.108901,2.672859,0.000000,0.4,1.0,0
2,dev_smoke_03_fixed_block_256_lru_random_fp16_4gib,10,random,fixed-block,lru,accelerator-fp16,2.211896,2.804167,0.000000,0.4,1.0,0
3,dev_smoke_04_radix_lru_random_fp16_4gib,10,random,radix,lru,accelerator-fp16,2.262065,2.854319,0.001283,0.4,1.0,0
4,dev_smoke_05_document_lru_random_cpu_fp16_4gib,10,random,document,lru,cpu-fp16,2.451631,3.155240,0.000000,0.4,1.0,0
5,dev_smoke_06_document_lru_random_int8_triton_4gib,10,random,document,lru,cpu-int8,2.512883,3.208204,0.000000,0.4,1.0,0
6,dev_smoke_07_segmented_zipf_fp16,10,zipf,none,none,accelerator-fp16,2.220218,2.484931,0.000000,0.7,NaN,0
7,dev_smoke_08_document_lru_zipf_fp16_4gib,10,zipf,document,lru,accelerator-fp16,1.587602,2.468748,0.298185,0.7,1.0,0
8,dev_smoke_09_fixed_block_256_lru_zipf_fp16_4gib,10,zipf,fixed-block,lru,accelerator-fp16,1.696347,2.629744,0.291891,0.7,1.0,0
9,dev_smoke_10_radix_lru_zipf_fp16_4gib,10,zipf,radix,lru,accelerator-fp16,1.627387,2.548180,0.298185,0.7,1.0,0


## 5. Full random trace, checkpoint 1/2

Runs 1--3: segmented reference, document FP16, and tuned fixed-block-256 FP16. The first run is intentionally the slowest because it computes every token without document reuse.

In [10]:
!python experiments/run_quality.py matrix configs/full_dev_confirmation.json --profile full --execute --resume --max-runs 3
show_summaries("full")

{
  "dataset": "data/quality-v1.0.1/QuALITY.v1.0.1.htmlstripped.dev",
  "split": "dev",
  "profile": "full",
  "no_inference": false,
  "matrix_type": "selected",
  "combinations": 3,
  "output_dir": "results/full_dev_confirmation/full",
  "execute": true
}
[1/3] skip results/full_dev_confirmation/full/dev_full_01_segmented_random_fp16.jsonl
[2/3] skip results/full_dev_confirmation/full/dev_full_02_document_lru_random_fp16_4gib.jsonl
[3/3] skip results/full_dev_confirmation/full/dev_full_03_fixed_block_256_lru_random_fp16_4gib.jsonl


,run,requests,workload,cache_strategy,policy,storage,ttft_mean_s,ttft_p95_s,article_token_hit_rate,accuracy,reference_label_agreement,reference_label_mismatches
0,dev_full_01_segmented_random_fp16,2086,random,none,none,accelerator-fp16,1.843387,2.889079,0.000000,0.580058,NaN,0
1,dev_full_02_document_lru_random_fp16_4gib,2086,random,document,lru,accelerator-fp16,1.439391,2.828208,0.229418,0.580058,1.000000,0
2,dev_full_03_fixed_block_256_lru_random_fp16_4gib,2086,random,fixed-block,lru,accelerator-fp16,1.473616,2.877259,0.231525,0.580537,0.999521,1
3,dev_full_04_radix_lru_random_fp16_4gib,2086,random,radix,lru,accelerator-fp16,1.528066,2.928337,0.227686,0.580058,1.000000,0
4,dev_full_05_document_lru_random_cpu_fp16_4gib,2086,random,document,lru,cpu-fp16,1.574088,2.993871,0.229418,0.580058,1.000000,0
5,dev_full_06_document_lru_random_int8_triton_4gib,2086,random,document,lru,cpu-int8,1.168059,2.907706,0.446737,0.580058,0.994247,12
6,dev_full_07_segmented_zipf_fp16,2086,zipf,none,none,accelerator-fp16,2.208352,2.733385,0.000000,0.564717,NaN,0
7,dev_full_08_document_lru_zipf_fp16_4gib,2086,zipf,document,lru,accelerator-fp16,0.805770,2.715824,0.653807,0.564717,1.000000,0
8,dev_full_09_fixed_block_256_lru_zipf_fp16_4gib,2086,zipf,fixed-block,lru,accelerator-fp16,0.863195,2.742370,0.652701,0.564717,0.998562,3


In [11]:
download_checkpoint("full-dev-checkpoint-03")

Created: /content/full-dev-checkpoint-03.zip (3.8 MiB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 6. Full random trace, checkpoint 2/2

Runs 4--6 add radix, CPU FP16 offload, and CPU INT8 with Triton restore. `--resume` skips runs 1--3. The random trace contains every dev question exactly once, so this is the suite's standard QuALITY accuracy result.

In [12]:
!python experiments/run_quality.py matrix configs/full_dev_confirmation.json --profile full --execute --resume --max-runs 6
show_summaries("full")

{
  "dataset": "data/quality-v1.0.1/QuALITY.v1.0.1.htmlstripped.dev",
  "split": "dev",
  "profile": "full",
  "no_inference": false,
  "matrix_type": "selected",
  "combinations": 6,
  "output_dir": "results/full_dev_confirmation/full",
  "execute": true
}
[1/6] skip results/full_dev_confirmation/full/dev_full_01_segmented_random_fp16.jsonl
[2/6] skip results/full_dev_confirmation/full/dev_full_02_document_lru_random_fp16_4gib.jsonl
[3/6] skip results/full_dev_confirmation/full/dev_full_03_fixed_block_256_lru_random_fp16_4gib.jsonl
[4/6] skip results/full_dev_confirmation/full/dev_full_04_radix_lru_random_fp16_4gib.jsonl
[5/6] skip results/full_dev_confirmation/full/dev_full_05_document_lru_random_cpu_fp16_4gib.jsonl
[6/6] skip results/full_dev_confirmation/full/dev_full_06_document_lru_random_int8_triton_4gib.jsonl


,run,requests,workload,cache_strategy,policy,storage,ttft_mean_s,ttft_p95_s,article_token_hit_rate,accuracy,reference_label_agreement,reference_label_mismatches
0,dev_full_01_segmented_random_fp16,2086,random,none,none,accelerator-fp16,1.843387,2.889079,0.000000,0.580058,NaN,0
1,dev_full_02_document_lru_random_fp16_4gib,2086,random,document,lru,accelerator-fp16,1.439391,2.828208,0.229418,0.580058,1.000000,0
2,dev_full_03_fixed_block_256_lru_random_fp16_4gib,2086,random,fixed-block,lru,accelerator-fp16,1.473616,2.877259,0.231525,0.580537,0.999521,1
3,dev_full_04_radix_lru_random_fp16_4gib,2086,random,radix,lru,accelerator-fp16,1.528066,2.928337,0.227686,0.580058,1.000000,0
4,dev_full_05_document_lru_random_cpu_fp16_4gib,2086,random,document,lru,cpu-fp16,1.574088,2.993871,0.229418,0.580058,1.000000,0
5,dev_full_06_document_lru_random_int8_triton_4gib,2086,random,document,lru,cpu-int8,1.168059,2.907706,0.446737,0.580058,0.994247,12
6,dev_full_07_segmented_zipf_fp16,2086,zipf,none,none,accelerator-fp16,2.208352,2.733385,0.000000,0.564717,NaN,0
7,dev_full_08_document_lru_zipf_fp16_4gib,2086,zipf,document,lru,accelerator-fp16,0.805770,2.715824,0.653807,0.564717,1.000000,0
8,dev_full_09_fixed_block_256_lru_zipf_fp16_4gib,2086,zipf,fixed-block,lru,accelerator-fp16,0.863195,2.742370,0.652701,0.564717,0.998562,3


In [13]:
download_checkpoint("full-dev-checkpoint-06-random-complete")

Created: /content/full-dev-checkpoint-06-random-complete.zip (3.8 MiB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 7. Full Zipf trace, checkpoint 1/2

Runs 7--9 create the aligned Zipf reference and compare document with fixed-block-256. Zipf repeats real questions according to article popularity; its accuracy is not the standard one-pass QuALITY accuracy.

In [14]:
!python experiments/run_quality.py matrix configs/full_dev_confirmation.json --profile full --execute --resume --max-runs 9
show_summaries("full")

{
  "dataset": "data/quality-v1.0.1/QuALITY.v1.0.1.htmlstripped.dev",
  "split": "dev",
  "profile": "full",
  "no_inference": false,
  "matrix_type": "selected",
  "combinations": 9,
  "output_dir": "results/full_dev_confirmation/full",
  "execute": true
}
[1/9] skip results/full_dev_confirmation/full/dev_full_01_segmented_random_fp16.jsonl
[2/9] skip results/full_dev_confirmation/full/dev_full_02_document_lru_random_fp16_4gib.jsonl
[3/9] skip results/full_dev_confirmation/full/dev_full_03_fixed_block_256_lru_random_fp16_4gib.jsonl
[4/9] skip results/full_dev_confirmation/full/dev_full_04_radix_lru_random_fp16_4gib.jsonl
[5/9] skip results/full_dev_confirmation/full/dev_full_05_document_lru_random_cpu_fp16_4gib.jsonl
[6/9] skip results/full_dev_confirmation/full/dev_full_06_document_lru_random_int8_triton_4gib.jsonl
[7/9] skip results/full_dev_confirmation/full/dev_full_07_segmented_zipf_fp16.jsonl
[8/9] skip results/full_dev_confirmation/full/dev_full_08_document_lru_zipf_fp16_4gib.j

,run,requests,workload,cache_strategy,policy,storage,ttft_mean_s,ttft_p95_s,article_token_hit_rate,accuracy,reference_label_agreement,reference_label_mismatches
0,dev_full_01_segmented_random_fp16,2086,random,none,none,accelerator-fp16,1.843387,2.889079,0.000000,0.580058,NaN,0
1,dev_full_02_document_lru_random_fp16_4gib,2086,random,document,lru,accelerator-fp16,1.439391,2.828208,0.229418,0.580058,1.000000,0
2,dev_full_03_fixed_block_256_lru_random_fp16_4gib,2086,random,fixed-block,lru,accelerator-fp16,1.473616,2.877259,0.231525,0.580537,0.999521,1
3,dev_full_04_radix_lru_random_fp16_4gib,2086,random,radix,lru,accelerator-fp16,1.528066,2.928337,0.227686,0.580058,1.000000,0
4,dev_full_05_document_lru_random_cpu_fp16_4gib,2086,random,document,lru,cpu-fp16,1.574088,2.993871,0.229418,0.580058,1.000000,0
5,dev_full_06_document_lru_random_int8_triton_4gib,2086,random,document,lru,cpu-int8,1.168059,2.907706,0.446737,0.580058,0.994247,12
6,dev_full_07_segmented_zipf_fp16,2086,zipf,none,none,accelerator-fp16,2.208352,2.733385,0.000000,0.564717,NaN,0
7,dev_full_08_document_lru_zipf_fp16_4gib,2086,zipf,document,lru,accelerator-fp16,0.805770,2.715824,0.653807,0.564717,1.000000,0
8,dev_full_09_fixed_block_256_lru_zipf_fp16_4gib,2086,zipf,fixed-block,lru,accelerator-fp16,0.863195,2.742370,0.652701,0.564717,0.998562,3


In [16]:
download_checkpoint("full-dev-checkpoint-09")

Created: /content/full-dev-checkpoint-09.zip (5.1 MiB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 8. Full Zipf trace, checkpoint 2/2

Runs 10--12 add radix, document GDSF, and CPU INT8/Triton. Re-running this cell is safe: completed JSONLs with their manifests are skipped.

In [15]:
!python experiments/run_quality.py matrix configs/full_dev_confirmation.json --profile full --execute --resume
show_summaries("full")

{
  "dataset": "data/quality-v1.0.1/QuALITY.v1.0.1.htmlstripped.dev",
  "split": "dev",
  "profile": "full",
  "no_inference": false,
  "matrix_type": "selected",
  "combinations": 12,
  "output_dir": "results/full_dev_confirmation/full",
  "execute": true
}
[1/12] skip results/full_dev_confirmation/full/dev_full_01_segmented_random_fp16.jsonl
[2/12] skip results/full_dev_confirmation/full/dev_full_02_document_lru_random_fp16_4gib.jsonl
[3/12] skip results/full_dev_confirmation/full/dev_full_03_fixed_block_256_lru_random_fp16_4gib.jsonl
[4/12] skip results/full_dev_confirmation/full/dev_full_04_radix_lru_random_fp16_4gib.jsonl
[5/12] skip results/full_dev_confirmation/full/dev_full_05_document_lru_random_cpu_fp16_4gib.jsonl
[6/12] skip results/full_dev_confirmation/full/dev_full_06_document_lru_random_int8_triton_4gib.jsonl
[7/12] skip results/full_dev_confirmation/full/dev_full_07_segmented_zipf_fp16.jsonl
[8/12] skip results/full_dev_confirmation/full/dev_full_08_document_lru_zipf_fp

,run,requests,workload,cache_strategy,policy,storage,ttft_mean_s,ttft_p95_s,article_token_hit_rate,accuracy,reference_label_agreement,reference_label_mismatches
0,dev_full_01_segmented_random_fp16,2086,random,none,none,accelerator-fp16,1.843387,2.889079,0.000000,0.580058,NaN,0
1,dev_full_02_document_lru_random_fp16_4gib,2086,random,document,lru,accelerator-fp16,1.439391,2.828208,0.229418,0.580058,1.000000,0
2,dev_full_03_fixed_block_256_lru_random_fp16_4gib,2086,random,fixed-block,lru,accelerator-fp16,1.473616,2.877259,0.231525,0.580537,0.999521,1
3,dev_full_04_radix_lru_random_fp16_4gib,2086,random,radix,lru,accelerator-fp16,1.528066,2.928337,0.227686,0.580058,1.000000,0
4,dev_full_05_document_lru_random_cpu_fp16_4gib,2086,random,document,lru,cpu-fp16,1.574088,2.993871,0.229418,0.580058,1.000000,0
5,dev_full_06_document_lru_random_int8_triton_4gib,2086,random,document,lru,cpu-int8,1.168059,2.907706,0.446737,0.580058,0.994247,12
6,dev_full_07_segmented_zipf_fp16,2086,zipf,none,none,accelerator-fp16,2.208352,2.733385,0.000000,0.564717,NaN,0
7,dev_full_08_document_lru_zipf_fp16_4gib,2086,zipf,document,lru,accelerator-fp16,0.805770,2.715824,0.653807,0.564717,1.000000,0
8,dev_full_09_fixed_block_256_lru_zipf_fp16_4gib,2086,zipf,fixed-block,lru,accelerator-fp16,0.863195,2.742370,0.652701,0.564717,0.998562,3
9,dev_full_10_radix_lru_zipf_fp16_4gib,2086,zipf,radix,lru,accelerator-fp16,0.762110,2.546098,0.652342,0.564717,1.000000,0


## 9. Validate, analyze, and download the final evidence

The analyzer rejects misaligned traces, incompatible schemas/manifests, and missing runs. It creates paired speedups, correctness diagnostics, tables, and figures.

In [17]:
!python experiments/run_quality.py analyze-inference results/full_dev_confirmation/full \
    --suite-config configs/full_dev_analysis.json \
    --output-dir results/full_dev_confirmation/analysis \
    --bootstrap-samples 20000 --seed 42

analysis_dir = Path("results/full_dev_confirmation/analysis")
display(pd.read_csv(analysis_dir / "run_summaries.csv"))
display(pd.read_csv(analysis_dir / "fair_speedups.csv"))
display(pd.read_csv(analysis_dir / "correctness.csv"))

results/full_dev_confirmation/analysis/run_summaries.csv
results/full_dev_confirmation/analysis/fair_speedups.csv
results/full_dev_confirmation/analysis/correctness.csv
results/full_dev_confirmation/analysis/mismatch_details.json
results/full_dev_confirmation/analysis/analysis.json
results/full_dev_confirmation/analysis/results.md
results/full_dev_confirmation/analysis/ttft_by_execution_path.pdf
results/full_dev_confirmation/analysis/ttft_by_execution_path.png
results/full_dev_confirmation/analysis/cache_only_speedup.pdf
results/full_dev_confirmation/analysis/cache_only_speedup.png
results/full_dev_confirmation/analysis/hit_latency_tradeoff.pdf
results/full_dev_confirmation/analysis/hit_latency_tradeoff.png


,accuracy,arena_metadata_bytes_peak,arena_peak_allocated_bytes,arena_reserved_bytes_peak,arena_stale_rejections,arena_stranded_bytes_peak,article_token_hit_rate,byte_hit_rate,cache_bytes_peak,dequant_mean_s,...,storage,store_mean_s,stranded_bytes_peak,strategy,transfer_mean_s,ttft_mean_s,ttft_p50_s,ttft_p95_s,useful_bytes_peak,workload
0,0.580058,0,0,0,0,0,0.000000,0.000000,0,0.000000,...,accelerator-fp16,0.000000,0,none,0.000000,1.843387,2.107362,2.889079,0,random
1,0.580058,0,0,0,0,0,0.229418,0.229418,4294950912,0.000000,...,accelerator-fp16,0.001206,0,document,0.000026,1.439391,1.598445,2.828208,4294950912,random
2,0.580537,0,0,0,0,0,0.231525,0.231525,4293918720,0.000000,...,accelerator-fp16,0.033433,0,fixed-block,0.000382,1.473616,1.558499,2.877259,4293918720,random
3,0.580058,0,0,0,0,0,0.227686,0.227686,4294778880,0.000000,...,accelerator-fp16,0.001352,0,radix,0.000077,1.528066,1.752034,2.928337,4294778880,random
4,0.580058,0,0,0,0,0,0.229418,0.229418,4294950912,0.000000,...,cpu-fp16,0.036954,0,document,0.009556,1.574088,1.773387,2.993871,4294950912,random
5,0.580058,0,0,0,0,0,0.446737,0.446737,4294874304,0.000943,...,cpu-int8,0.029801,0,document,0.011173,1.168059,0.786471,2.907706,4294874304,random
6,0.564717,0,0,0,0,0,0.000000,0.000000,0,0.000000,...,accelerator-fp16,0.000000,0,none,0.000000,2.208352,2.323056,2.733385,0,zipf
7,0.564717,0,0,0,0,0,0.653807,0.653807,4294406144,0.000000,...,accelerator-fp16,0.000673,0,document,0.000069,0.805770,0.068047,2.715824,4294406144,zipf
8,0.564717,0,0,0,0,0,0.652701,0.652701,4293918720,0.000000,...,accelerator-fp16,0.034215,0,fixed-block,0.001101,0.863195,0.185028,2.742370,4293918720,zipf
9,0.564717,0,0,0,0,0,0.652342,0.652342,4294922240,0.000000,...,accelerator-fp16,0.000626,0,radix,0.000112,0.762110,0.068294,2.546098,4294922240,zipf


,article_token_hit_rate,byte_hit_rate,cache_bytes_peak,cache_only_speedup,cache_only_speedup_ci95_high,cache_only_speedup_ci95_low,cache_only_ttft_reduction_percent,cached_ttft_mean_s,cached_ttft_p50_s,cached_ttft_p95_s,...,label,policy,run,segmentation_speedup,segmented_ttft_mean_s,segmented_ttft_p50_s,segmented_ttft_p95_s,storage,strategy,workload
0,0.229418,0.229418,4294950912,1.280672,1.312976,1.250398,21.915988,1.439391,1.598445,2.828208,...,Document LRU accelerator FP16,lru,02_document_lru_random_fp16_4gib,NaN,1.843387,2.107362,2.889079,accelerator-fp16,document,random
1,0.231525,0.231525,4293918720,1.250928,1.280545,1.223035,20.059324,1.473616,1.558499,2.877259,...,Fixed-block 256 LRU accelerator FP16,lru,03_fixed_block_256_lru_random_fp16_4gib,NaN,1.843387,2.107362,2.889079,accelerator-fp16,fixed-block,random
2,0.227686,0.227686,4294778880,1.206353,1.236391,1.178110,17.105518,1.528066,1.752034,2.928337,...,Radix LRU accelerator FP16,lru,04_radix_lru_random_fp16_4gib,NaN,1.843387,2.107362,2.889079,accelerator-fp16,radix,random
3,0.229418,0.229418,4294950912,1.171083,1.199730,1.144355,14.608926,1.574088,1.773387,2.993871,...,Document LRU CPU FP16,lru,05_document_lru_random_cpu_fp16_4gib,NaN,1.843387,2.107362,2.889079,cpu-fp16,document,random
4,0.446737,0.446737,4294874304,1.578162,1.642781,1.519682,36.635165,1.168059,0.786471,2.907706,...,"Document LRU CPU INT8, Triton restore",lru,06_document_lru_random_int8_triton_4gib,NaN,1.843387,2.107362,2.889079,cpu-int8,document,random
5,0.653807,0.653807,4294406144,2.740674,2.902935,2.594045,63.512626,0.805770,0.068047,2.715824,...,Document LRU accelerator FP16,lru,08_document_lru_zipf_fp16_4gib,NaN,2.208352,2.323056,2.733385,accelerator-fp16,document,zipf
6,0.652701,0.652701,4293918720,2.558346,2.692624,2.436976,60.912240,0.863195,0.185028,2.742370,...,Fixed-block 256 LRU accelerator FP16,lru,09_fixed_block_256_lru_zipf_fp16_4gib,NaN,2.208352,2.323056,2.733385,accelerator-fp16,fixed-block,zipf
7,0.652342,0.652342,4294922240,2.897683,3.068493,2.744864,65.489666,0.762110,0.068294,2.546098,...,Radix LRU accelerator FP16,lru,10_radix_lru_zipf_fp16_4gib,NaN,2.208352,2.323056,2.733385,accelerator-fp16,radix,zipf
8,0.714567,0.714567,4294692864,3.486676,3.720698,3.279674,71.319392,0.633369,0.064850,2.539322,...,Document GDSF accelerator FP16,gdsf,11_document_gdsf_zipf_fp16_4gib,NaN,2.208352,2.323056,2.733385,accelerator-fp16,document,zipf
9,0.816640,0.816640,4294959424,4.851154,5.251499,4.503397,79.386347,0.455222,0.086603,2.563011,...,"Document LRU CPU INT8, Triton restore",lru,12_document_lru_zipf_int8_triton_4gib,NaN,2.208352,2.323056,2.733385,cpu-int8,document,zipf


,accuracy_delta_vs_segmented,agreement_vs_segmented,cached_accuracy,cached_hard_accuracy,hard_accuracy_delta_vs_segmented,label,label_mismatches,max_label_logit_delta_vs_segmented,mean_max_label_logit_delta_vs_segmented,run,segmented_accuracy,segmented_hard_accuracy,storage,unique_label_mismatches,workload
0,0.000000,1.000000,0.580058,0.489202,0.000000,Document LRU accelerator FP16,0,0.000000,0.000000,02_document_lru_random_fp16_4gib,0.580058,0.489202,accelerator-fp16,0,random
1,0.000479,0.999521,0.580537,0.489202,0.000000,Fixed-block 256 LRU accelerator FP16,1,0.125000,0.006524,03_fixed_block_256_lru_random_fp16_4gib,0.580058,0.489202,accelerator-fp16,1,random
2,0.000000,1.000000,0.580058,0.489202,0.000000,Radix LRU accelerator FP16,0,0.093750,0.002824,04_radix_lru_random_fp16_4gib,0.580058,0.489202,accelerator-fp16,0,random
3,0.000000,1.000000,0.580058,0.489202,0.000000,Document LRU CPU FP16,0,0.000000,0.000000,05_document_lru_random_cpu_fp16_4gib,0.580058,0.489202,cpu-fp16,0,random
4,0.000000,0.994247,0.580058,0.487324,-0.001878,"Document LRU CPU INT8, Triton restore",12,1.312500,0.114611,06_document_lru_random_int8_triton_4gib,0.580058,0.489202,cpu-int8,12,random
5,0.000000,1.000000,0.564717,0.511554,0.000000,Document LRU accelerator FP16,0,0.000000,0.000000,08_document_lru_zipf_fp16_4gib,0.564717,0.511554,accelerator-fp16,0,zipf
6,0.000000,0.998562,0.564717,0.511554,0.000000,Fixed-block 256 LRU accelerator FP16,3,0.125000,0.018614,09_fixed_block_256_lru_zipf_fp16_4gib,0.564717,0.511554,accelerator-fp16,2,zipf
7,0.000000,1.000000,0.564717,0.511554,0.000000,Radix LRU accelerator FP16,0,0.078125,0.002052,10_radix_lru_zipf_fp16_4gib,0.564717,0.511554,accelerator-fp16,0,zipf
8,0.000000,1.000000,0.564717,0.511554,0.000000,Document GDSF accelerator FP16,0,0.000000,0.000000,11_document_gdsf_zipf_fp16_4gib,0.564717,0.511554,accelerator-fp16,0,zipf
9,0.009108,0.972675,0.573826,0.527490,0.015936,"Document LRU CPU INT8, Triton restore",57,1.515625,0.227559,12_document_lru_zipf_int8_triton_4gib,0.564717,0.511554,cpu-int8,14,zipf


In [18]:
download_checkpoint(
    "full-dev-confirmation-complete",
    source=Path("results/full_dev_confirmation"),
)

Created: /content/full-dev-confirmation-complete.zip (5.5 MiB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

The browser downloads do not use Google Drive quota. Verify each ZIP locally before ending a runtime. Keep the raw JSONL, summaries, and manifests together; only curated generated analysis should later be committed to Git.